In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
import pygeohash as pgh

def time_to_minutes(t):
    if pd.isnull(t): return np.nan
    h, m = map(int, str(t).split(':'))
    return h * 60 + m

def create_safe_features(train_path, test_path):
    print("1. Loading datasets...")
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df['is_test'] = 0
    test_df['is_test'] = 1

    combined = pd.concat([train_df, test_df], ignore_index=True)
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else np.nan)
    combined['minutes'] = combined['timestamp'].apply(time_to_minutes)

    # ==========================================
    # 1. Base Spatial & Temporal Features
    # ==========================================
    print("2. Engineering Base Features...")
    combined['lat'] = combined['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    combined['lon'] = combined['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)
    combined['time_sin'] = np.sin(2 * np.pi * combined['minutes'] / 1440)
    combined['time_cos'] = np.cos(2 * np.pi * combined['minutes'] / 1440)

    combined['LargeVehicles'] = combined['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(-1)
    combined['Landmarks'] = combined['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(-1)
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].median())

    # ==========================================
    # 2. Spatial Clustering
    # ==========================================
    print("3. K-Means Spatial Clustering...")
    coords = combined[['lat', 'lon']].dropna()
    kmeans = KMeans(n_clusters=30, random_state=42, n_init=10)
    kmeans.fit(coords)
    combined.loc[coords.index, 'spatial_cluster'] = kmeans.labels_
    combined['spatial_cluster'] = combined['spatial_cluster'].fillna(-1).astype(int)

    # ==========================================
    # 3. 100% SAFE Historical Target Encoding
    # ==========================================
    print("4. Calculating Safe Historical Demand (Train-Only)...")
    # We strictly isolate train to prevent NaN collapse in test
    train_only = combined[combined['is_test'] == 0]

    geo_means = train_only.groupby('geohash')['demand'].mean().rename('hist_geo_mean')
    geo_hour_means = train_only.groupby(['geohash', 'hour'])['demand'].mean().rename('hist_geo_hour_mean')
    cluster_means = train_only.groupby('spatial_cluster')['demand'].mean().rename('hist_cluster_mean')

    # Map back to combined
    combined = combined.merge(geo_means, on='geohash', how='left')
    combined = combined.merge(geo_hour_means, on=['geohash', 'hour'], how='left')
    combined = combined.merge(cluster_means, on='spatial_cluster', how='left')

    # Safely fill gaps for any brand new geohashes in test set
    global_mean = train_only['demand'].mean()
    combined['hist_geo_mean'] = combined['hist_geo_mean'].fillna(global_mean)
    combined['hist_geo_hour_mean'] = combined['hist_geo_hour_mean'].fillna(combined['hist_geo_mean'])
    combined['hist_cluster_mean'] = combined['hist_cluster_mean'].fillna(global_mean)

    # ==========================================
    # 4. Final Processing
    # ==========================================
    print("5. Processing Categoricals...")
    cat_features = ['geohash', 'RoadType', 'Weather']
    for col in cat_features:
        combined[col] = combined[col].fillna('Missing').astype(str)

    train_full = combined[combined['is_test'] == 0].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'hour'])
    test_full = combined[combined['is_test'] == 1].drop(columns=['is_test', 'Index', 'timestamp', 'minutes', 'hour', 'demand'])

    xgb_train = train_full.copy()
    xgb_test = test_full.copy()
    for col in cat_features:
        le = LabelEncoder()
        le.fit(combined[col])
        xgb_train[col] = le.transform(xgb_train[col])
        xgb_test[col] = le.transform(xgb_test[col])

    return train_full, test_full, xgb_train, xgb_test, cat_features

if __name__ == "__main__":
    cb_train, cb_test, xgb_train, xgb_test, cat_cols = create_safe_features('dataset/train.csv', 'dataset/test.csv')

    max_day = cb_train['day'].max()
    y_full = cb_train['demand']

    train_mask = cb_train['day'] < max_day
    val_mask = cb_train['day'] == max_day
    y_tr, y_va = y_full[train_mask], y_full[val_mask]

    # ---------------------------------------------------------
    # TRAIN CATBOOST
    # ---------------------------------------------------------
    print("\n--- Training CatBoost Model ---")
    X_cb_tr = cb_train[train_mask].drop(columns=['day', 'demand'])
    X_cb_va = cb_train[val_mask].drop(columns=['day', 'demand'])
    X_cb_full = cb_train.drop(columns=['day', 'demand'])
    X_cb_test = cb_test.drop(columns=['day'])

    cb_model = CatBoostRegressor(
        iterations=4000, learning_rate=0.015, depth=6,
        task_type='GPU', eval_metric='RMSE', random_seed=42
    )
    cb_model.fit(X_cb_tr, y_tr, eval_set=(X_cb_va, y_va), cat_features=cat_cols, early_stopping_rounds=150, verbose=500)

    print("Retraining CatBoost on 100% data...")
    cb_final = CatBoostRegressor(
        iterations=cb_model.get_best_iteration(), learning_rate=0.015, depth=6,
        task_type='GPU', random_seed=42
    )
    cb_final.fit(X_cb_full, y_full, cat_features=cat_cols, verbose=False)
    cb_preds = cb_final.predict(X_cb_test)

    # ---------------------------------------------------------
    # TRAIN XGBOOST
    # ---------------------------------------------------------
    print("\n--- Training XGBoost Model ---")
    X_xgb_tr = xgb_train[train_mask].drop(columns=['day', 'demand'])
    X_xgb_va = xgb_train[val_mask].drop(columns=['day', 'demand'])
    X_xgb_full = xgb_train.drop(columns=['day', 'demand'])
    X_xgb_test = xgb_test.drop(columns=['day'])

    dtrain = xgb.DMatrix(X_xgb_tr, label=y_tr)
    dval = xgb.DMatrix(X_xgb_va, label=y_va)

    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'device': 'cuda',
        'learning_rate': 0.015,
        'max_depth': 6,
        'min_child_weight': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.7
    }
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=4000, evals=[(dval, 'Validation')], early_stopping_rounds=150, verbose_eval=500)

    print("Retraining XGBoost on 100% data...")
    dtrain_full = xgb.DMatrix(X_xgb_full, label=y_full)
    dtest = xgb.DMatrix(X_xgb_test)
    xgb_final = xgb.train(xgb_params, dtrain_full, num_boost_round=xgb_model.best_iteration)
    xgb_preds = xgb_final.predict(dtest)

    # ---------------------------------------------------------
    # ENSEMBLE & SAVE (MATCHES YOUR OPTUNA SCRIPT)
    # ---------------------------------------------------------
    print("\n--- Finalizing Predictions ---")
    cb_preds = np.clip(cb_preds, 0, None)
    xgb_preds = np.clip(xgb_preds, 0, None)

    final_predictions = (cb_preds * 0.5) + (xgb_preds * 0.5)

    test_raw = pd.read_csv('dataset/test.csv')

    sub = pd.DataFrame({
        'Index': test_raw['Index'],
        'demand': final_predictions
    })

    sub.to_csv('submission_safe_features7.csv', index=False)
    print("Done! Saved 'submission_safe_features7.csv'")

1. Loading datasets...
2. Engineering Base Features...
3. K-Means Spatial Clustering...
4. Calculating Safe Historical Demand (Train-Only)...
5. Processing Categoricals...

--- Training CatBoost Model ---
0:	learn: 0.1398395	test: 0.1437123	best: 0.1437123 (0)	total: 17.2ms	remaining: 1m 8s
bestTest = 0.05941821172
bestIteration = 196
Shrink model to first 197 iterations.
Retraining CatBoost on 100% data...

--- Training XGBoost Model ---
[0]	Validation-rmse:0.14364
[385]	Validation-rmse:0.06277
Retraining XGBoost on 100% data...

--- Finalizing Predictions ---
Done! Saved 'submission_safe_features7.csv'
